# Annex: Normalisation and the Learning Rate

A reading annex for **Problems 4** (sections 4.2 and 4.3). It uses a different dataset from the problems, so you can run it freely and then apply the same ideas there.

We answer three questions:

1. Why do features on different scales make gradient descent difficult?
2. How do we normalise them (and what do outliers do)?
3. How do we use a model trained on normalised data?

**The data** is the `mpg` car dataset from Session 1. We predict the highway consumption `hwy` (miles per gallon) from the engine size `displ` (litres) and the model `year`. Notice the scales: `displ` is around 1.6-7, `year` is around 2000.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

url = "https://raw.githubusercontent.com/cienciadadesdsuab/DS26-27/main/Laboratoris/01_Introduction%20to%20Python/mpg.csv"
mpg = pd.read_csv(url, index_col=0)

features = mpg[["displ", "year"]].to_numpy(dtype=float)
hwy = mpg["hwy"].to_numpy(dtype=float)
mpg[["displ", "year", "hwy"]].describe().loc[["min", "max", "mean", "std"]]

## 1. The gradient descent function

The same vectorised function from the problems (see also the *Annex: Gradient Descent Function*). The only change is that it takes the design matrix `X` and returns the cost $J = \frac{1}{2m}\sum (\hat y - y)^2$ at every iteration so we can watch the training.

In [ ]:
def gradient_descent(X, y, alpha, iterations=200):
    m, n = X.shape
    w = np.zeros(n)
    J = np.zeros(iterations)
    with np.errstate(all="ignore"):               # we want to see divergence, not warnings
        for it in range(iterations):
            error = X @ w - y
            J[it] = error @ error / (2 * m)       # cost before the update
            w = w - alpha * (X.T @ error) / m     # gradient step
    return w, J

ones = np.ones((len(hwy), 1))
X_raw = np.hstack([ones, features])               # columns: 1, displ, year
X_raw[:3]

## 2. Without normalising

Gradient descent moves every weight by `alpha × gradient`. The gradient of the weight of `year` is about **2000 times larger** than that of the bias (the column of ones), because each term is multiplied by the feature value. Let's try three learning rates:

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 3.5))
for a, alpha in zip(ax, [1e-1, 1e-6, 1e-8]):
    w, J = gradient_descent(X_raw, hwy, alpha, iterations=200)
    a.plot(J)
    a.set_title(f"alpha = {alpha:g}\nfinal cost = {J[-1]:.3g}")
    a.set_xlabel("iteration"); a.set_ylabel("cost")
plt.show()

* **alpha = 0.1**: far too large for the `year` column. Each step overshoots and the cost grows without limit (it ends up `nan`): *divergence*.
* **alpha = 1e-6**: still too large for `year`, and it diverges too.
* **alpha = 1e-8**: it does not diverge, and the cost curve even *looks* converged. But the final cost (about 17.7) is **much higher** than the best achievable (about 7, see section 4). The step is so tiny for the small columns (the bias and `displ`) that they have hardly moved, and the curve flattens far from the optimum.

There is **no single alpha** that is good for both the small and the huge column. The cost surface is a long, narrow valley, and gradient descent zigzags along it.

## 3. Normalising the features

If all columns are of order 1, a single alpha suits all of them. Two common ways:

**Min-max scaling** (to $[0,1]$): $\quad x' = \dfrac{x - x_{min}}{x_{max} - x_{min}}$

**Standardisation** (z-score): $\quad x' = \dfrac{x - \mu}{\sigma}$, which gives mean 0 and standard deviation 1.

Which one? Look at what an **outlier** does to each on a tiny example:

In [ ]:
x = np.array([10., 12., 11., 13., 12., 11., 10., 14., 500.])     # the last value is an outlier

minmax = (x - x.min()) / (x.max() - x.min())
zscore = (x - x.mean()) / x.std()

print("min-max:", np.round(minmax, 3))
print("z-score:", np.round(zscore, 2))

* With **min-max**, the outlier becomes 1 and *all the normal values are squeezed together* near 0 (they differ in the third decimal).
* With **z-score** the outlier also dominates the mean and the standard deviation, so the normal values are squeezed too: both methods are sensitive to outliers. Check for them *before* normalising, and decide whether to remove them (e.g. with a boolean mask).

For regression we usually standardise. Important rule: **compute $\mu$ and $\sigma$ on the training data once, and reuse them for any new data.** Never recompute them on the new points.

Let's standardise `displ` and `year` (not the column of ones!):

In [ ]:
mu = features.mean(axis=0)
sigma = features.std(axis=0)

Z = (features - mu) / sigma
X_std = np.hstack([ones, Z])

print("means after :", np.round(Z.mean(axis=0), 3))
print("stds after  :", np.round(Z.std(axis=0), 3))

## 4. Gradient descent on the normalised data

Now the same function with a much larger learning rate:

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 3.5))
for a, alpha in zip(ax, [0.01, 0.1, 2.0]):
    w, J = gradient_descent(X_std, hwy, alpha, iterations=200)
    a.plot(J)
    a.set_title(f"alpha = {alpha:g}\nfinal cost = {J[-1]:.3g}")
    a.set_xlabel("iteration"); a.set_ylabel("cost")
plt.show()

* **alpha = 0.01**: converges, but slowly (a cautious step).
* **alpha = 0.1**: converges in a few dozen iterations. This is the kind of value that works with normalised data.
* **alpha = 2**: too large (the limit here is about 1.7), the cost explodes again.

So normalising does not remove the need to choose alpha, but it makes a *reasonable* alpha (0.01-1) exist, and it makes convergence **much faster**.

**Reading the cost plot** is your main diagnostic: it must go down smoothly. If it goes up or oscillates, reduce alpha. If it goes down very slowly, increase alpha.

## 5. Using the model: predictions

The weights we obtain belong to the **normalised** features. To predict for a new car (say 3.0 litres, year 2008) we must normalise it **with the same $\mu$ and $\sigma$ as the training data**, add the 1 for the bias, and take the dot product:

In [ ]:
w, J = gradient_descent(X_std, hwy, alpha=0.1, iterations=500)

new_car = np.array([3.0, 2008.0])
z_new = (new_car - mu) / sigma                 # SAME mu and sigma as in training
x_new = np.r_[1.0, z_new]

print("weights (normalised features):", np.round(w, 3))
print("predicted hwy:", round(float(x_new @ w), 1), "mpg")

Let's check that gradient descent found the exact least-squares solution:

In [ ]:
w_exact, *_ = np.linalg.lstsq(X_std, hwy, rcond=None)
print("gradient descent:", np.round(w, 4))
print("least squares   :", np.round(w_exact, 4))

# Summary

| Question | Answer |
|---|---|
| Why normalise? | Features on very different scales make the cost surface a narrow valley: no single alpha works and convergence is very slow or diverges. |
| How? | Standardise: $(x - \mu)/\sigma$, or min-max to $[0,1]$. Never touch the column of ones. |
| Outliers | They distort both methods: look for them first. |
| New data | Normalise with the **training** $\mu$ and $\sigma$, never with the new data's. |
| Choosing alpha | Watch the cost curve: it should decrease smoothly. Diverges → smaller alpha. Too slow → larger alpha. |

Now go back to **Problems 4 (sections 4.2 and 4.3)** and apply the same ideas to the house prices.